# MT5 Portfolio Analyzer

Runs the four-pair grid strategy portfolio reconstruction from MT5 backtest exports and displays all key results interactively.

**What this notebook does:**
1. Runs the simulation engine against your XLSX + tester-graph CSV files
2. Displays a combined balance & equity curve over time
3. Shows drawdown profile
4. Shows per-pair cumulative PnL contribution
5. Displays a summary metrics table

**Data directory:** `data/`  
**Config:** `config/example_config.json`

In [ ]:
import sys, os

# Resolve repo root robustly whether notebook is run from repo root or notebooks/
cwd = os.getcwd()
candidates = [cwd, os.path.dirname(cwd)]
REPO = next((p for p in candidates if os.path.exists(os.path.join(p, "src"))), cwd)
SRC = os.path.join(REPO, "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)

import warnings
warnings.filterwarnings("ignore")

print(f"Repo root : {REPO}")
print(f"Python    : {sys.executable}")

import pandas as pd
import plotly
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
from plotly.subplots import make_subplots

# Import analyzer types and functions
from mt5_portfolio_analyzer import (
    DealEvent, TradeEvent, PairData, ScenarioConfig,
    run_simulation, _interpolate_floating,
)

# nbformat was installed after the kernel started - re-inject it so plotly sees it
import importlib
import nbformat
import plotly.io._renderers as _plotly_renderers
_plotly_renderers.nbformat = nbformat

pio.renderers.default = "notebook_connected"

print("pandas", pd.__version__, "| plotly", plotly.__version__,
      "| nbformat", nbformat.__version__, "| renderer:", pio.renderers.default)

## Step 1 - Run the Simulation

Loads all four XLSX reports and tester-graph CSVs from `data/`, runs the combined portfolio simulation, and stores the results in memory.

In [ ]:
from IPython.display import display
import importlib
import numpy as np
import mt5_readers as mtr
import mt5_portfolio_analyzer as mpa

# Ensure notebook picks up latest reader/engine edits without a manual kernel restart.
importlib.reload(mtr)
importlib.reload(mpa)

from mt5_readers import discover_files
from mt5_portfolio_analyzer import (
    load_pair, run_simulation,
    discover_reference_price_files,
)

# Use baseline scenario (update to 'proposed' to run proposed scenario)
DATA_DIR = os.path.join(REPO, "data", "baseline")

# --- Simulation parameters (edit as needed) ---
INITIAL_BALANCE = 5000.0
SCALE_EXPONENT  = 1.0
MIN_SCALE       = 0.1
MAX_SCALE       = 5.0

# --- Broker margin requirements ---
MMR_CSV = os.path.join(REPO, "data", "reference", "forex_com_margin_requirements.csv")
margin_requirements = {}
if os.path.exists(MMR_CSV):
    mmr_df = pd.read_csv(MMR_CSV)
    for _, row in mmr_df.iterrows():
        pair_key = "".join(ch for ch in str(row["currency_pair"]).upper() if ch.isalpha())
        margin_requirements[pair_key] = float(row["mmr_percent"])
    print(f"Loaded MMR for {len(margin_requirements)} pairs from {MMR_CSV}")
else:
    print("MMR file not found, margin metrics will be blank.")


def average_monthly_balance_growth(curve_rows):
    df = pd.DataFrame(curve_rows)
    if df.empty:
        return None
    df["time"] = pd.to_datetime(df["time"], format="%Y.%m.%d %H:%M")
    df["ym"] = df["time"].dt.to_period("M")
    monthly_bal = df.groupby("ym")["balance"].last()
    if len(monthly_bal) < 2:
        return None
    returns = monthly_bal.pct_change().dropna()
    return returns.mean() * 100 if len(returns) > 0 else None


def recompute_shared_dd(curve_rows):
    """Recompute DD using definition: drawdown = floating_pnl / balance."""
    df = pd.DataFrame(curve_rows)
    if df.empty or "equity" not in df.columns or "floating_pnl" not in df.columns:
        return {
            "max_drawdown_abs": 0.0,
            "max_drawdown_percent": 0.0,
            "max_drawdown_time": None,
            "max_drawdown_peak_equity": None,
            "equity_at_max_drawdown": None,
        }

    if "time" in df.columns:
        df = df.copy()
        df["time"] = pd.to_datetime(df["time"], format="%Y.%m.%d %H:%M", errors="coerce")
        df = df.sort_values("time").reset_index(drop=True)

    equity = pd.to_numeric(df["equity"], errors="coerce").fillna(0.0).to_numpy()
    floating = pd.to_numeric(df["floating_pnl"], errors="coerce").fillna(0.0).to_numpy()
    balance = equity - floating
    if len(equity) == 0:
        return {
            "max_drawdown_abs": 0.0,
            "max_drawdown_percent": 0.0,
            "max_drawdown_time": None,
            "max_drawdown_peak_equity": None,
            "equity_at_max_drawdown": None,
        }

    dd_ratio = np.where(balance != 0, floating / balance, 0.0)
    dd_abs = floating
    dd_pct = dd_ratio * 100.0

    dd_idx = int(np.argmin(dd_ratio))
    dd_time = None
    if "time" in df.columns and dd_idx < len(df):
        t = df.iloc[dd_idx]["time"]
        dd_time = t.strftime("%Y.%m.%d %H:%M") if pd.notna(t) else None

    return {
        "max_drawdown_abs": float(dd_abs[dd_idx]),
        "max_drawdown_percent": float(dd_pct[dd_idx]),
        "max_drawdown_time": dd_time,
        "max_drawdown_peak_equity": None,
        "equity_at_max_drawdown": float(equity[dd_idx]),
    }


def apply_shared_dd(summary_dict, curve_rows):
    out = dict(summary_dict) if summary_dict is not None else {}
    out.update(recompute_shared_dd(curve_rows))
    return out



# --- Discover optional M1 market data used for floating PnL reconstruction ---
REFERENCE_DIR = os.path.join(REPO, "data", "reference")
market_files = discover_reference_price_files(REFERENCE_DIR) if os.path.isdir(REFERENCE_DIR) else {}
if market_files:
    print(f"Loaded M1 market mappings for {len(market_files)} pairs from {REFERENCE_DIR}")
else:
    print("No M1 market files found; floating/equity may be flat or zero.")

# --- Load all pairs and infer baseline profile from MT5 reports ---
discovered = discover_files(DATA_DIR)
pairs_data = []
baseline_inferred_config = {}
validation_rows = []
for pair, files in discovered.items():
    xlsx = files.get("xlsx")
    csv  = files.get("csv")
    market_csv = market_files.get(pair)
    if not xlsx:
        print(f"  SKIP {pair}: no xlsx found")
        continue

    print(f"  Loading {pair} ...")
    loaded_pair = load_pair(
        name=pair,
        xlsx_path=xlsx,
        csv_path=csv,
        market_csv_path=market_csv,
    )
    pairs_data.append(loaded_pair)

    baseline_inferred_config[pair] = {
        "risk": float(loaded_pair.baseline_config.risk_percent),
        "tp": loaded_pair.baseline_config.take_profit,
        "grid": loaded_pair.baseline_config.grid_size,
        "max_trades": int(loaded_pair.baseline_config.max_trades),
        "initial_balance": float(loaded_pair.baseline_config.initial_balance),
        "first_lot": loaded_pair.baseline_config.first_lot,
        "median_lot": loaded_pair.baseline_config.median_lot,
        "trade_count": int(loaded_pair.baseline_config.trade_count),
    }

    validation_rows.append({
        "Pair": pair,
        "Risk": round(float(loaded_pair.baseline_config.risk_percent), 4),
        "TP": loaded_pair.baseline_config.take_profit,
        "Grid": loaded_pair.baseline_config.grid_size,
        "Max Trades": int(loaded_pair.baseline_config.max_trades),
        "Initial Balance": float(loaded_pair.baseline_config.initial_balance),
        "First Lot": loaded_pair.baseline_config.first_lot,
        "Median Lot": loaded_pair.baseline_config.median_lot,
        "Trades": int(loaded_pair.baseline_config.trade_count),
    })

if validation_rows:
    df_inference_validation = pd.DataFrame(validation_rows).sort_values("Pair")
    print("\nValidation Report: Inferred Baseline EA Configuration from MT5")
    display(df_inference_validation)

# --- Run combined simulation ---
result = run_simulation(
    pairs_data=pairs_data,
    initial_balance=INITIAL_BALANCE,
    scale_exponent=SCALE_EXPONENT,
    min_scale=MIN_SCALE,
    max_scale=MAX_SCALE,
    margin_requirements=margin_requirements,
)

summary = apply_shared_dd(result["summary"], result["curve_rows"])
result["summary"] = summary
baseline_avg_monthly_growth = average_monthly_balance_growth(result["curve_rows"])

print(f"\nDone. {summary['total_deals']} deals | "
      f"{summary['period_start']} -> {summary['period_end']}")
if baseline_avg_monthly_growth is not None:
    print(f"Avg Monthly Balance Growth: {baseline_avg_monthly_growth:.3f}%")

## Step 2 - Summary Metrics Table

In [ ]:
from IPython.display import display

if "baseline_avg_monthly_growth" not in globals():
    baseline_avg_monthly_growth = average_monthly_balance_growth(result["curve_rows"])

# --- Portfolio-level summary ---
portfolio_metrics = {
    "Metric": [
        "Initial Balance",
        "Final Balance",
        "Total Return",
        "CAGR",
        "Avg Monthly Balance Growth",
        "Max Drawdown (abs)",
        "Max Drawdown (%)",
        "Max Used Margin",
        "Min Free Margin",
        "Min Margin Level (%)",
        "Total Deals",
        "Period",
    ],
    "Value": [
        f"${summary['initial_balance']:,.2f}",
        f"${summary['final_balance']:,.2f}",
        f"{summary['total_return_percent']:.2f}%",
        f"{summary['cagr_percent']:.2f}%",
        f"{baseline_avg_monthly_growth:.3f}%" if baseline_avg_monthly_growth is not None else "N/A",
        f"${summary['max_drawdown_abs']:,.2f}",
        f"{summary['max_drawdown_percent']:.2f}%",
        f"${summary['max_used_margin']:,.2f}" if summary.get('max_used_margin') is not None else "N/A",
        f"${summary['min_free_margin']:,.2f}" if summary.get('min_free_margin') is not None else "N/A",
        f"{summary['min_margin_level_percent']:.2f}%" if summary.get('min_margin_level_percent') is not None else "N/A",
        str(summary["total_deals"]),
        f"{summary['period_start']}  ->  {summary['period_end']}",
    ],
}
df_summary = pd.DataFrame(portfolio_metrics)
display(df_summary.style.hide(axis="index").set_caption("Portfolio Summary"))

# --- Per-pair breakdown ---
pair_rows = []
for pair, info in summary["pairs"].items():
    pair_rows.append({
        "Pair":            pair,
        "Risk %":          info["risk_percent"],
        "Deals":           info["deals_count"],
        "Median Vol":      info["baseline_volume_median"],
        "Scaled PnL ($)":  f"{info['scaled_pnl_contribution']:+,.2f}",
    })
df_pairs = pd.DataFrame(pair_rows)
display(df_pairs.style.hide(axis="index").set_caption("Per-Pair Contribution"))

## Step 3 - Combined Balance, Equity & Drawdown

In [ ]:
df_curve = pd.DataFrame(result["curve_rows"])
df_curve["time"] = pd.to_datetime(df_curve["time"], format="%Y.%m.%d %H:%M")

balance = pd.to_numeric(df_curve["balance"], errors="coerce")
floating = pd.to_numeric(df_curve["floating_pnl"], errors="coerce")

dd_abs = floating
dd_pct = np.where(balance != 0, (floating / balance) * 100.0, 0.0)

fig = make_subplots(
    rows=3, cols=1,
    shared_xaxes=True,
    row_heights=[0.52, 0.24, 0.24],
    subplot_titles=["Combined Balance & Equity", "Floating PnL (USD)", "Drawdown = Floating/Balance (%)"],
    vertical_spacing=0.06,
)

fig.add_trace(go.Scatter(
    x=df_curve["time"], y=df_curve["balance"],
    name="Balance", line=dict(color="#2196F3", width=1.6),
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=df_curve["time"], y=df_curve["equity"],
    name="Equity", line=dict(color="#4CAF50", width=1.2, dash="dot"),
    fill="tonexty", fillcolor="rgba(76,175,80,0.08)",
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=df_curve["time"], y=dd_abs,
    fill="tozeroy", fillcolor="rgba(244,67,54,0.20)",
    line=dict(color="#F44336", width=1),
    name="Floating PnL ($)",
), row=2, col=1)

fig.add_trace(go.Scatter(
    x=df_curve["time"], y=dd_pct,
    fill="tozeroy", fillcolor="rgba(244,67,54,0.15)",
    line=dict(color="#E91E63", width=1),
    name="DD = Floating/Balance (%)",
), row=3, col=1)

fig.add_hline(
    y=INITIAL_BALANCE,
    line_dash="dash",
    line_color="gray",
    annotation_text=f"Start ${INITIAL_BALANCE:,.0f}",
    row=1,
    col=1,
)

fig.update_layout(
    height=760,
    title=f"Portfolio Reconstruction  |  {summary['period_start'][:10]} -> {summary['period_end'][:10]}",
    hovermode="x unified",
    legend=dict(orientation="h", y=1.03),
    margin=dict(t=90),
)
fig.update_yaxes(title_text="USD", tickprefix="$", row=1, col=1)
fig.update_yaxes(title_text="USD", tickprefix="$", row=2, col=1)
fig.update_yaxes(title_text="%", ticksuffix="%", row=3, col=1)
fig.show()

## Step 5 - Per-Pair Cumulative PnL Contribution

In [ ]:
df_events = pd.DataFrame(result["event_rows"])
df_events["time"] = pd.to_datetime(df_events["time"], format="%Y.%m.%d %H:%M:%S")

PAIR_COLORS = {
    "EURUSD": "#2196F3",
    "EURGBP": "#9C27B0",
    "GBPUSD": "#4CAF50",
    "USDCHF": "#FF9800",
}

# Cumulative PnL per pair
fig3 = go.Figure()
for pair in df_events["pair"].unique():
    sub = df_events[df_events["pair"] == pair].copy()
    sub = sub.sort_values("time")
    sub["cum_pnl"] = sub["scaled_net_profit"].cumsum()
    fig3.add_trace(go.Scatter(
        x=sub["time"],
        y=sub["cum_pnl"],
        name=pair,
        mode="lines",
        line=dict(color=PAIR_COLORS.get(pair), width=1.8),
    ))

fig3.update_layout(
    title="Cumulative Scaled PnL by Pair",
    height=420,
    hovermode="x unified",
    yaxis=dict(title="Cumulative PnL ($)", tickprefix="$"),
    legend=dict(orientation="h", y=1.04),
    margin=dict(t=80),
)
fig3.show()

# Per-pair floating PnL over time (baseline reconstruction)
df_curve_floating = pd.DataFrame(result["curve_rows"]).copy()
df_curve_floating["time"] = pd.to_datetime(df_curve_floating["time"], format="%Y.%m.%d %H:%M")

def _pair_multiplier(pair_data):
    cfg = pair_data.config
    if cfg.base_lot is not None and pair_data.baseline_volume_median and pair_data.baseline_volume_median > 0:
        return cfg.base_lot / pair_data.baseline_volume_median
    return 1.0

fig5 = go.Figure()
for pair_data in pairs_data:
    pair = pair_data.config.name
    pm = _pair_multiplier(pair_data)
    scale = pd.to_numeric(df_curve_floating["balance"], errors="coerce").fillna(0.0) / INITIAL_BALANCE
    floating_series = [
        _interpolate_floating(pair_data.curve, t) * float(s) * pm
        for t, s in zip(df_curve_floating["time"], scale)
    ]

    fig5.add_trace(go.Scatter(
        x=df_curve_floating["time"],
        y=floating_series,
        name=pair,
        mode="lines",
        line=dict(color=PAIR_COLORS.get(pair), width=1.6),
    ))

fig5.update_layout(
    title="Per-Pair Floating PnL by Pair",
    height=420,
    hovermode="x unified",
    yaxis=dict(title="Floating PnL ($)", tickprefix="$"),
    legend=dict(orientation="h", y=1.04),
    margin=dict(t=80),
)
fig5.show()

# Pie chart - final contribution share
pairs_info = summary["pairs"]
fig4 = px.pie(
    names=list(pairs_info.keys()),
    values=[max(0, v["scaled_pnl_contribution"]) for v in pairs_info.values()],
    color=list(pairs_info.keys()),
    color_discrete_map=PAIR_COLORS,
    title="PnL Contribution Share",
    hole=0.4,
)
fig4.update_traces(textinfo="label+percent")
fig4.update_layout(height=380, margin=dict(t=60))
fig4.show()

## Step 7 - Scenario Configuration & Synthesis

**Edit the configuration table below to modify pair parameters. The analyzer will synthesize results by:**
- **Risk %**: Scale volumes/profits proportionally
- **Max Trades**: Limit to first N closed trades  
- **TP/Grid**: Displayed for reference (strategy parameters)

In [ ]:
# Configuration baseline (inferred from MT5 backtests in Step 1)
import numpy as np

DEFAULT_GRID_BY_PAIR = {
    'EURUSD': 450,
    'EURGBP': 450,
    'GBPUSD': 500,
    'USDCHF': 400,
}

baseline_config = {}
for pair in ['EURUSD', 'EURGBP', 'GBPUSD', 'USDCHF']:
    inferred = baseline_inferred_config.get(pair, {}) if 'baseline_inferred_config' in globals() else {}

    baseline_config[pair] = {
        'risk': float(inferred.get('risk', 0.0)),
        'tp': int(inferred['tp']) if inferred.get('tp') is not None else 10,
        'grid': int(inferred['grid']) if inferred.get('grid') is not None else int(DEFAULT_GRID_BY_PAIR[pair]),
        'max_trades': int(inferred['max_trades']) if inferred.get('max_trades') else 1,
    }

pairs_by_name = {p.name: p for p in pairs_data}

# Display baseline configuration table (inferred)
config_rows = []
for pair in ['EURUSD', 'EURGBP', 'GBPUSD', 'USDCHF']:
    cfg = baseline_config[pair]
    config_rows.append({
        'Pair': pair,
        'Risk % (inferred)': cfg['risk'],
        'TP (inferred pips)': cfg['tp'],
        'Grid (default)': cfg['grid'],
        'Max Trades (inferred)': cfg['max_trades'],
    })
df_config = pd.DataFrame(config_rows)
display(df_config)

if 'df_inference_validation' in globals():
    print("\nValidation Report (from MT5 source data):")
    display(df_inference_validation)

print("\n" + "=" * 80)
print("SCENARIO PARAMETERS - Edit below to change pair configuration")
print("=" * 80)

# Scenario: Modify parameters here
scenario_config = {
    # Default scenario keeps inferred baseline except EURUSD override example.
    'EURUSD':  {'risk': 3.0, 'tp': baseline_config['EURUSD']['tp'], 'grid': 450, 'max_trades': 4},
    'EURGBP':  {'risk': baseline_config['EURGBP']['risk'], 'tp': baseline_config['EURGBP']['tp'], 'grid': 450, 'max_trades': baseline_config['EURGBP']['max_trades']},
    'GBPUSD':  {'risk': baseline_config['GBPUSD']['risk'], 'tp': baseline_config['GBPUSD']['tp'], 'grid': 500, 'max_trades': baseline_config['GBPUSD']['max_trades']},
    'USDCHF':  {'risk': baseline_config['USDCHF']['risk'], 'tp': baseline_config['USDCHF']['tp'], 'grid': 400, 'max_trades': baseline_config['USDCHF']['max_trades']},
}

# Scenario synthesis mode:
# - 'scaling_only'       : apply risk scaling only; structural params are ignored for synthesis.
# - 'approx_structural'  : apply risk + structural approximations (TP/Grid/MaxTrades) with caveats.
SYNTHESIS_MODE = 'approx_structural'

STRUCTURAL_KEYS = ('tp', 'grid', 'max_trades')

def resolve_effective_cfg(base_cfg, scen_cfg, mode='scaling_only'):
    eff = dict(scen_cfg)
    structural_changed = any(scen_cfg[k] != base_cfg[k] for k in STRUCTURAL_KEYS)
    ignored = []

    if mode == 'scaling_only':
        for k in STRUCTURAL_KEYS:
            if eff[k] != base_cfg[k]:
                ignored.append(k)
                eff[k] = base_cfg[k]

    return eff, structural_changed, ignored

def synthesize_pair_events(orig_pair, base_cfg, scen_cfg):
    """
    Build a synthesized pair from baseline deals/trades by:
    1) scaling included trades linearly by risk,
    2) enforcing scenario max-trades as a hard open-trade cap (approximation),
    3) scaling closed deal PnL for included legs (approximation for TP/Grid),
    4) reusing baseline pair curve; floating is scaled in run_simulation from reconstructed positions.
    """
    risk_factor = scen_cfg['risk'] / base_cfg['risk'] if base_cfg['risk'] > 0 else 1.0
    tp_factor = scen_cfg['tp'] / base_cfg['tp'] if base_cfg['tp'] > 0 else 1.0
    grid_factor = base_cfg['grid'] / scen_cfg['grid'] if scen_cfg['grid'] > 0 else 1.0
    pnl_factor = risk_factor * tp_factor * grid_factor

    base_max = int(base_cfg['max_trades']) if base_cfg['max_trades'] > 0 else 1
    scen_max = int(scen_cfg['max_trades']) if scen_cfg['max_trades'] > 0 else 1

    trades_sorted = sorted(orig_pair.trades, key=lambda t: t.time)

    modified_deals = []
    modified_trades = []

    # Maintain FIFO ledger so each out closes the corresponding earlier in.
    open_keep_flags = []
    kept_open = 0
    deal_idx = 0

    for tr in trades_sorted:
        if tr.direction == 'in':
            keep_leg = kept_open < scen_max
            open_keep_flags.append(keep_leg)
            if keep_leg:
                kept_open += 1
                modified_trades.append(TradeEvent(
                    time=tr.time,
                    pair=tr.pair,
                    direction=tr.direction,
                    volume=tr.volume * risk_factor,
                    price=tr.price,
                ))

        elif tr.direction == 'out':
            keep_leg = open_keep_flags.pop(0) if open_keep_flags else True

            deal = None
            if deal_idx < len(orig_pair.deals):
                deal = orig_pair.deals[deal_idx]
                deal_idx += 1

            if keep_leg:
                kept_open = max(0, kept_open - 1)
                modified_trades.append(TradeEvent(
                    time=tr.time,
                    pair=tr.pair,
                    direction=tr.direction,
                    volume=tr.volume * risk_factor,
                    price=tr.price,
                ))
                if deal is not None:
                    modified_deals.append(DealEvent(
                        time=deal.time,
                        pair=deal.pair,
                        net_profit=deal.net_profit * pnl_factor,
                        volume=deal.volume * risk_factor,
                    ))

    # IMPORTANT: do not pre-scale floating curve here.
    # The simulator now scales floating from reconstructed open positions;
    # pre-scaling here would apply scenario factors twice.
    synth_curve = list(orig_pair.curve)

    return modified_deals, modified_trades, synth_curve, {
        'risk_factor': risk_factor,
        'tp_factor': tp_factor,
        'grid_factor': grid_factor,
        'pnl_factor': pnl_factor,
        'base_max_trades': base_max,
        'scenario_max_trades': scen_max,
        'baseline_deals': len(orig_pair.deals),
        'deals_kept': len(modified_deals),
    }


print(f"\nSynthesis mode: {SYNTHESIS_MODE}")
if SYNTHESIS_MODE == 'scaling_only':
    print("Structural params (TP/Grid/MaxTrades) are ignored for synthesis and should be validated with real MT5 backtests.")
else:
    print("Structural params are being approximated from baseline history (not an exact EA re-run).")

# Build synthesized scenario pairs
scenario_pairs_data = []
scenario_factor_rows = []
ignored_structural_rows = []

for pair in ['EURUSD', 'EURGBP', 'GBPUSD', 'USDCHF']:
    if pair not in pairs_by_name:
        continue

    orig_pair = pairs_by_name[pair]
    base_cfg = baseline_config[pair]
    raw_scen_cfg = scenario_config[pair]
    scen_cfg, structural_changed, ignored_keys = resolve_effective_cfg(base_cfg, raw_scen_cfg, SYNTHESIS_MODE)

    synth_deals, synth_trades, synth_curve, factors = synthesize_pair_events(orig_pair, base_cfg, scen_cfg)

    synth_pair = PairData(
        name=pair,
        baseline_config=orig_pair.baseline_config,
        deals=synth_deals,
        trades=synth_trades,
        curve=synth_curve,
        baseline_volume_median=orig_pair.baseline_volume_median,
        scenario_config=ScenarioConfig(
            risk_percent=scen_cfg['risk'],
            take_profit=scen_cfg['tp'],
            grid_size=scen_cfg['grid'],
            max_trades=scen_cfg['max_trades'],
        ),
        market_times=orig_pair.market_times,
        market_close=orig_pair.market_close,
    )
    scenario_pairs_data.append(synth_pair)

    scenario_factor_rows.append({
        'Pair': pair,
        'Risk x': round(factors['risk_factor'], 4),
        'TP x': round(factors['tp_factor'], 4),
        'Grid x': round(factors['grid_factor'], 4),
        'PnL x': round(factors['pnl_factor'], 4),
        'MaxTrades (B->Eff)': f"{factors['base_max_trades']}->{factors['scenario_max_trades']}",
        'Deals baseline': factors['baseline_deals'],
        'Deals kept': factors['deals_kept'],
        'Deals dropped': factors['baseline_deals'] - factors['deals_kept'],
        'Structural changed?': structural_changed,
        'Structural applied?': len(ignored_keys) == 0,
    })

    if ignored_keys:
        ignored_structural_rows.append({
            'Pair': pair,
            'Ignored keys': ', '.join(ignored_keys),
            'Configured values': str({k: raw_scen_cfg[k] for k in ignored_keys}),
            'Effective values': str({k: base_cfg[k] for k in ignored_keys}),
        })

print("\nSynthesis factors used:")
display(pd.DataFrame(scenario_factor_rows))

if ignored_structural_rows:
    print("\nStructural parameters ignored in scaling_only mode:")
    display(pd.DataFrame(ignored_structural_rows))

# Run synthesized scenario simulation
SCENARIO_BALANCE = 5000.0
SCENARIO_EXPONENT = 1.0
SCENARIO_MIN_SCALE = 0.1
SCENARIO_MAX_SCALE = 5.0

r2 = run_simulation(
    pairs_data=scenario_pairs_data,
    initial_balance=SCENARIO_BALANCE,
    scale_exponent=SCENARIO_EXPONENT,
    min_scale=SCENARIO_MIN_SCALE,
    max_scale=SCENARIO_MAX_SCALE,
    margin_requirements=margin_requirements,
)
s2 = apply_shared_dd(r2['summary'], r2['curve_rows'])
r2['summary'] = s2
scenario_avg_monthly_growth = average_monthly_balance_growth(r2['curve_rows'])

print(f"\nSynthesized scenario generated: {len(r2['event_rows'])} deals")
print(
    f"Return {s2['total_return_percent']:.2f}% | "
    f"MaxDD {s2['max_drawdown_percent']:.2f}% | "
    f"CAGR {s2['cagr_percent']:.2f}% | "
    f"Avg Monthly {scenario_avg_monthly_growth:.3f}%"
)

df_c2 = pd.DataFrame(r2['curve_rows'])
df_c2['time'] = pd.to_datetime(df_c2['time'], format='%Y.%m.%d %H:%M')
peak2 = df_c2['equity'].cummax()
dd2 = (peak2 - df_c2['equity']) / peak2 * 100

fig6 = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.65, 0.35],
    subplot_titles=['Balance', 'Drawdown (%)'],
    vertical_spacing=0.06,
)

fig6.add_trace(go.Scatter(
    x=df_c2['time'], y=df_c2['balance'],
    name='Scenario Balance', line=dict(color='#2196F3')
), row=1, col=1)
fig6.add_trace(go.Scatter(
    x=df_c2['time'], y=df_c2['equity'],
    name='Scenario Equity', line=dict(color='#4CAF50', dash='dot')
), row=1, col=1)
fig6.add_trace(go.Scatter(
    x=df_c2['time'], y=-dd2,
    fill='tozeroy', fillcolor='rgba(244,67,54,0.18)',
    line=dict(color='#F44336'), name='Scenario DD %'
), row=2, col=1)

fig6.update_layout(
    title=(
        f"Synthesized Scenario ({SYNTHESIS_MODE}) | Return {s2['total_return_percent']:.1f}% | "
        f"MaxDD {s2['max_drawdown_percent']:.1f}% | "
        f"CAGR {s2['cagr_percent']:.1f}% | "
        f"Avg Monthly {scenario_avg_monthly_growth:.3f}%"
    ),
    height=520,
    hovermode='x unified',
    legend=dict(orientation='h', y=1.04),
    margin=dict(t=80),
)
fig6.update_yaxes(tickprefix='$', row=1, col=1)
fig6.update_yaxes(ticksuffix='%', row=2, col=1)
fig6.show()

In [ ]:
# Find the time of maximum drawdown and show per-pair status at that moment
import numpy as np

balance_vals = pd.to_numeric(df_curve["balance"], errors="coerce").fillna(0.0).values
floating_vals = pd.to_numeric(df_curve["floating_pnl"], errors="coerce").fillna(0.0).values
equity_vals = pd.to_numeric(df_curve["equity"], errors="coerce").fillna(0.0).values

dd_pct_vals = np.where(balance_vals != 0, (floating_vals / balance_vals) * 100.0, 0.0)

# Find worst drawdown index and time (most negative ratio)
max_dd_idx = np.argmin(dd_pct_vals)
max_dd_time = df_curve.iloc[max_dd_idx]["time"]
max_dd_value = dd_pct_vals[max_dd_idx]
max_dd_abs = floating_vals[max_dd_idx]

print(f"Maximum Drawdown Event:")
print(f"  Date: {max_dd_time}")
print(f"  DD %: {max_dd_value:.2f}%")
print(f"  DD $: ${max_dd_abs:,.2f}")
print(f"  Current Equity: ${equity_vals[max_dd_idx]:,.2f}")
print(f"  Balance: ${balance_vals[max_dd_idx]:,.2f}")
print(f"  Floating PnL: ${floating_vals[max_dd_idx]:,.2f}")

# Show per-pair baseline floating snapshot at that time (diagnostic)
print(f"\n--- Per-Pair Baseline Floating Snapshot at {max_dd_time.strftime('%Y-%m-%d %H:%M')} ---")
for pair_data in pairs_data:
    from mt5_portfolio_analyzer import _interpolate_floating
    float_pnl = _interpolate_floating(pair_data.curve, max_dd_time)
    print(f"\n{pair_data.config.name}:")
    print(f"  Baseline floating snapshot: ${float_pnl:,.2f}")

## Step 7b - Scenario vs Baseline Comparison

In [ ]:
# Display scenario configuration vs baseline
print("=" * 80)
print("SCENARIO CONFIGURATION (Baseline -> Synthesized Scenario)")
print("=" * 80)

scenario_rows = []
for pair in ['EURUSD', 'EURGBP', 'GBPUSD', 'USDCHF']:
    b = baseline_config[pair]
    s = scenario_config[pair]
    scenario_rows.append({
        'Pair': pair,
        'Risk % (B->S)': f"{b['risk']} -> {s['risk']}",
        'TP (B->S)': f"{b['tp']} -> {s['tp']}",
        'Grid (B->S)': f"{b['grid']} -> {s['grid']}",
        'Max Trades (B->S)': f"{b['max_trades']} -> {s['max_trades']}",
    })

df_scenario_config = pd.DataFrame(scenario_rows)
display(df_scenario_config)

print("\n" + "=" * 80)
print("SYNTHESIS SUMMARY")
print("=" * 80)
print(f"Baseline deals: {len(result['event_rows'])}")
print(f"Synthesized scenario deals: {len(r2['event_rows'])}")

baseline_monthly_growth = average_monthly_balance_growth(result['curve_rows'])
scenario_monthly_growth = average_monthly_balance_growth(r2['curve_rows'])


def calc_min_margin(df_in):
    """Minimum MT4/MT5 margin level %: equity / used_margin * 100."""
    if df_in is None:
        return None

    if 'margin_level_percent' in df_in.columns:
        margin = pd.to_numeric(df_in['margin_level_percent'], errors='coerce')
        margin = margin.replace([np.inf, -np.inf], np.nan).dropna()
        return margin.min() if len(margin) > 0 else None

    if 'used_margin' in df_in.columns and 'equity' in df_in.columns:
        used = pd.to_numeric(df_in['used_margin'], errors='coerce')
        eq = pd.to_numeric(df_in['equity'], errors='coerce')
        margin = (eq / used.replace(0, np.nan) * 100).replace([np.inf, -np.inf], np.nan).dropna()
        return margin.min() if len(margin) > 0 else None

    return None


baseline_margin = calc_min_margin(pd.DataFrame(result['curve_rows']))
scenario_margin = calc_min_margin(pd.DataFrame(r2['curve_rows']))

# Optional validation against actual proposed backtest files
proposed_validation = None
proposed_validation_monthly = None
proposed_validation_margin = None
PROPOSED_DIR = os.path.join(REPO, 'data', 'proposed')
if os.path.exists(PROPOSED_DIR) and len(os.listdir(PROPOSED_DIR)) > 0:
    discovered_proposed = discover_files(PROPOSED_DIR)
    pairs_data_proposed = []
    for pair, files in discovered_proposed.items():
        xlsx = files.get('xlsx')
        csv = files.get('csv')
        if not xlsx:
            continue
        pairs_data_proposed.append(load_pair(
            name=pair,
            xlsx_path=xlsx,
            csv_path=csv,
        ))

    if pairs_data_proposed:
        proposed_result = run_simulation(
            pairs_data=pairs_data_proposed,
            initial_balance=INITIAL_BALANCE,
            scale_exponent=SCALE_EXPONENT,
            min_scale=MIN_SCALE,
            max_scale=MAX_SCALE,
            margin_requirements=margin_requirements,
        )
        proposed_validation = apply_shared_dd(proposed_result['summary'], proposed_result['curve_rows'])
        proposed_validation_monthly = average_monthly_balance_growth(proposed_result['curve_rows'])
        proposed_validation_margin = calc_min_margin(pd.DataFrame(proposed_result['curve_rows']))
        print("Validation using actual proposed backtests loaded.")

# Compare baseline vs synthesized, and optionally vs actual proposed
comparison_data = {
    'Metric': [
        'Final Balance',
        'Total Return (%)',
        'CAGR (%)',
        'Avg Monthly Balance Growth (%)',
        'Min Margin Level (%)',
        'Max Drawdown ($)',
        'Max Drawdown (%)',
        'Final Equity',
    ],
    'Baseline': [
        f"${summary['final_balance']:,.2f}",
        f"{summary['total_return_percent']:.2f}%",
        f"{summary['cagr_percent']:.2f}%",
        f"{baseline_monthly_growth:.3f}%" if baseline_monthly_growth is not None else 'N/A',
        f"{baseline_margin:.2f}%" if baseline_margin is not None else 'N/A',
        f"${summary['max_drawdown_abs']:,.2f}",
        f"{summary['max_drawdown_percent']:.2f}%",
        f"${summary['final_equity']:,.2f}",
    ],
    'Synthesized Scenario': [
        f"${s2['final_balance']:,.2f}",
        f"{s2['total_return_percent']:.2f}%",
        f"{s2['cagr_percent']:.2f}%",
        f"{scenario_monthly_growth:.3f}%" if scenario_monthly_growth is not None else 'N/A',
        f"{scenario_margin:.2f}%" if scenario_margin is not None else 'N/A',
        f"${s2['max_drawdown_abs']:,.2f}",
        f"{s2['max_drawdown_percent']:.2f}%",
        f"${s2['final_equity']:,.2f}",
    ],
    'Delta vs Baseline': [
        f"${s2['final_balance'] - summary['final_balance']:+,.2f}",
        f"{s2['total_return_percent'] - summary['total_return_percent']:+.2f}pp",
        f"{s2['cagr_percent'] - summary['cagr_percent']:+.2f}pp",
        (
            f"{scenario_monthly_growth - baseline_monthly_growth:+.3f}pp"
            if (scenario_monthly_growth is not None and baseline_monthly_growth is not None)
            else 'N/A'
        ),
        (
            f"{scenario_margin - baseline_margin:+.2f}pp"
            if (scenario_margin is not None and baseline_margin is not None)
            else 'N/A'
        ),
        f"${s2['max_drawdown_abs'] - summary['max_drawdown_abs']:+,.2f}",
        f"{s2['max_drawdown_percent'] - summary['max_drawdown_percent']:+.2f}pp",
        f"${s2['final_equity'] - summary['final_equity']:+,.2f}",
    ],
}

if proposed_validation is not None:
    comparison_data['Actual Proposed'] = [
        f"${proposed_validation['final_balance']:,.2f}",
        f"{proposed_validation['total_return_percent']:.2f}%",
        f"{proposed_validation['cagr_percent']:.2f}%",
        f"{proposed_validation_monthly:.3f}%" if proposed_validation_monthly is not None else 'N/A',
        f"{proposed_validation_margin:.2f}%" if proposed_validation_margin is not None else 'N/A',
        f"${proposed_validation['max_drawdown_abs']:,.2f}",
        f"{proposed_validation['max_drawdown_percent']:.2f}%",
        f"${proposed_validation['final_equity']:,.2f}",
    ]
    comparison_data['Synth Error vs Actual'] = [
        f"${s2['final_balance'] - proposed_validation['final_balance']:+,.2f}",
        f"{s2['total_return_percent'] - proposed_validation['total_return_percent']:+.2f}pp",
        f"{s2['cagr_percent'] - proposed_validation['cagr_percent']:+.2f}pp",
        (
            f"{scenario_monthly_growth - proposed_validation_monthly:+.3f}pp"
            if (scenario_monthly_growth is not None and proposed_validation_monthly is not None)
            else 'N/A'
        ),
        (
            f"{scenario_margin - proposed_validation_margin:+.2f}pp"
            if (scenario_margin is not None and proposed_validation_margin is not None)
            else 'N/A'
        ),
        f"${s2['max_drawdown_abs'] - proposed_validation['max_drawdown_abs']:+,.2f}",
        f"{s2['max_drawdown_percent'] - proposed_validation['max_drawdown_percent']:+.2f}pp",
        f"${s2['final_equity'] - proposed_validation['final_equity']:+,.2f}",
    ]

df_synth_comp = pd.DataFrame(comparison_data)
display(df_synth_comp)

In [ ]:
# Step 7b.1 - EURUSD Risk Sensitivity Sanity Check (isolated vs mixed)
# This isolates EURUSD risk-only impact so it is not conflated with max_trades/other-pair changes.

def _run_scenario_with_config(cfg_map):
    local_pairs = []
    for _pair in ['EURUSD', 'EURGBP', 'GBPUSD', 'USDCHF']:
        if _pair not in pairs_by_name:
            continue
        _orig = pairs_by_name[_pair]
        _base = baseline_config[_pair]
        _scen = cfg_map[_pair]
        _deals, _trades, _curve, _factors = synthesize_pair_events(_orig, _base, _scen)
        local_pairs.append(PairData(
            name=_pair,
            baseline_config=_orig.baseline_config,
            deals=_deals,
            trades=_trades,
            curve=_curve,
            baseline_volume_median=_orig.baseline_volume_median,
            scenario_config=ScenarioConfig(
                risk_percent=_scen['risk'],
                take_profit=_scen['tp'],
                grid_size=_scen['grid'],
                max_trades=_scen['max_trades'],
            ),
            market_times=_orig.market_times,
            market_close=_orig.market_close,
        ))

    _r = run_simulation(
        pairs_data=local_pairs,
        initial_balance=SCENARIO_BALANCE,
        scale_exponent=SCENARIO_EXPONENT,
        min_scale=SCENARIO_MIN_SCALE,
        max_scale=SCENARIO_MAX_SCALE,
        margin_requirements=margin_requirements,
    )
    _s = apply_shared_dd(_r['summary'], _r['curve_rows'])
    _r['summary'] = _s
    _monthly = average_monthly_balance_growth(_r['curve_rows'])
    return _r, _s, _monthly


# Mixed scenario already computed in r2/s2/scenario_monthly_growth
base_eu = summary['pairs']['EURUSD']['scaled_pnl_contribution']
mixed_eu = s2['pairs']['EURUSD']['scaled_pnl_contribution']

# Isolated scenario: only EURUSD risk 3.3 -> 3.0, keep all other knobs at baseline.
isolated_cfg = {k: dict(v) for k, v in baseline_config.items()}
isolated_cfg['EURUSD']['risk'] = 3.0

r_iso, s_iso, monthly_iso = _run_scenario_with_config(isolated_cfg)
iso_eu = s_iso['pairs']['EURUSD']['scaled_pnl_contribution']

risk_linear_factor = isolated_cfg['EURUSD']['risk'] / baseline_config['EURUSD']['risk']

sanity_rows = [
    {
        'Case': 'EURUSD Contribution - Baseline',
        'Value': f"${base_eu:,.2f}",
    },
    {
        'Case': 'EURUSD Contribution - Mixed Scenario (Step 7)',
        'Value': f"${mixed_eu:,.2f} ({(mixed_eu / base_eu - 1) * 100:+.2f}%)",
    },
    {
        'Case': 'EURUSD Contribution - Isolated Risk-Only (3.3 -> 3.0)',
        'Value': f"${iso_eu:,.2f} ({(iso_eu / base_eu - 1) * 100:+.2f}%)",
    },
    {
        'Case': 'Expected Linear Risk Factor (3.0/3.3)',
        'Value': f"{risk_linear_factor:.4f} ({(risk_linear_factor - 1) * 100:+.2f}%)",
    },
    {
        'Case': 'Portfolio Avg Monthly - Baseline',
        'Value': f"{baseline_monthly_growth:.3f}%" if baseline_monthly_growth is not None else 'N/A',
    },
    {
        'Case': 'Portfolio Avg Monthly - Mixed Scenario',
        'Value': f"{scenario_monthly_growth:.3f}% ({scenario_monthly_growth - baseline_monthly_growth:+.3f}pp)"
                 if (scenario_monthly_growth is not None and baseline_monthly_growth is not None) else 'N/A',
    },
    {
        'Case': 'Portfolio Avg Monthly - Isolated EURUSD Risk-Only',
        'Value': f"{monthly_iso:.3f}% ({monthly_iso - baseline_monthly_growth:+.3f}pp)"
                 if (monthly_iso is not None and baseline_monthly_growth is not None) else 'N/A',
    },
]

display(pd.DataFrame(sanity_rows))

In [ ]:
# Step 7c - Max Drawdown Forensics (Baseline vs Synthesized)
import numpy as np


def _pair_multiplier(pair_data):
    cfg = pair_data.config
    if cfg.base_lot is not None and pair_data.baseline_volume_median and pair_data.baseline_volume_median > 0:
        return cfg.base_lot / pair_data.baseline_volume_median
    return 1.0


def _scenario_dd_context(curve_rows, pair_data_list, label, initial_balance):
    """Return max-DD context and per-pair floating contribution at DD timestamp."""
    df = pd.DataFrame(curve_rows).copy()
    df['time'] = pd.to_datetime(df['time'], format='%Y.%m.%d %H:%M')

    equity_vals = pd.to_numeric(df['equity'], errors='coerce').fillna(0.0).to_numpy()
    floating_vals = pd.to_numeric(df['floating_pnl'], errors='coerce').fillna(0.0).to_numpy()
    balance_vals = pd.to_numeric(df['balance'], errors='coerce').fillna(0.0).to_numpy()
    dd_abs_vals = floating_vals
    dd_pct_vals = np.where(balance_vals != 0, dd_abs_vals / balance_vals * 100.0, 0.0)

    dd_idx = int(np.argmin(dd_pct_vals))
    dd_time = df.iloc[dd_idx]['time']
    dd_pct = float(dd_pct_vals[dd_idx])
    dd_abs = float(dd_abs_vals[dd_idx])
    peak_eq = None
    cur_eq = float(equity_vals[dd_idx])

    bal_idx = max(0, np.searchsorted(df['time'].to_numpy(), dd_time) - 1)
    bal_at_time = float(df.iloc[bal_idx]['balance'])
    bscale = bal_at_time / initial_balance if initial_balance > 0 else 1.0

    rows = []
    total_float = 0.0
    for pair_data in pair_data_list:
        pm = _pair_multiplier(pair_data)
        base_float = _interpolate_floating(pair_data.curve, dd_time)
        scaled_float = base_float * bscale * pm
        total_float += scaled_float
        rows.append({
            'Scenario': label,
            'DD Timestamp': dd_time.strftime('%Y-%m-%d %H:%M'),
            'Pair': pair_data.config.name,
            'Base Floating PnL': base_float,
            'Balance Scale': bscale,
            'Pair Multiplier': pm,
            'Scaled Floating PnL': scaled_float,
        })

    summary = {
        'Scenario': label,
        'DD Timestamp': dd_time.strftime('%Y-%m-%d %H:%M'),
        'Max DD %': dd_pct,
        'Max DD $': dd_abs,
        'Peak Equity': peak_eq,
        'Equity @ DD': cur_eq,
        'Balance @ DD': bal_at_time,
        'Total Floating @ DD': total_float,
    }
    return summary, rows


baseline_dd_summary, baseline_dd_rows = _scenario_dd_context(
    result['curve_rows'],
    pairs_data,
    'Baseline',
    INITIAL_BALANCE,
)

synth_dd_summary, synth_dd_rows = _scenario_dd_context(
    r2['curve_rows'],
    scenario_pairs_data,
    'Synthesized',
    INITIAL_BALANCE,
)

print('Max Drawdown Context (Portfolio-Level)')
df_dd_summary = pd.DataFrame([baseline_dd_summary, synth_dd_summary])
for col in ['Max DD %', 'Max DD $', 'Peak Equity', 'Equity @ DD', 'Balance @ DD', 'Total Floating @ DD']:
    df_dd_summary[col] = pd.to_numeric(df_dd_summary[col], errors='coerce')
display(df_dd_summary)

print('\nPer-Pair Floating Contribution At Each Scenario\'s DD Timestamp')
df_dd_pairs = pd.DataFrame(baseline_dd_rows + synth_dd_rows)
for col in ['Base Floating PnL', 'Balance Scale', 'Pair Multiplier', 'Scaled Floating PnL']:
    df_dd_pairs[col] = pd.to_numeric(df_dd_pairs[col], errors='coerce')

display(
    df_dd_pairs.sort_values(['Scenario', 'Pair'])
)

print('\nPer-Pair Delta In Scaled Floating PnL (Synth - Baseline)')
base_p = pd.DataFrame(baseline_dd_rows)[['Pair', 'Scaled Floating PnL']].rename(columns={'Scaled Floating PnL': 'Baseline Scaled Floating'})
syn_p = pd.DataFrame(synth_dd_rows)[['Pair', 'Scaled Floating PnL']].rename(columns={'Scaled Floating PnL': 'Synth Scaled Floating'})
delta_p = base_p.merge(syn_p, on='Pair', how='outer')
delta_p['Delta (Synth - Baseline)'] = delta_p['Synth Scaled Floating'] - delta_p['Baseline Scaled Floating']
display(delta_p.sort_values('Pair'))

## Step 8 - Baseline vs Proposed Comparison

Compare key metrics between two scenarios: baseline and proposed strategy parameters.
Add backtest files to `data/proposed/` folder to enable this comparison.


In [ ]:
import os
import numpy as np

# Load baseline (already loaded above as result/summary)
baseline_summary = apply_shared_dd(summary, result['curve_rows'])
baseline_result = result
baseline_df_curve = pd.DataFrame(result['curve_rows'])
baseline_df_curve['time'] = pd.to_datetime(baseline_df_curve['time'], format='%Y.%m.%d %H:%M')

# Try to load proposed scenario
PROPOSED_DIR = os.path.join(REPO, 'data', 'proposed')
proposed_summary = None
proposed_result = None
proposed_df_curve = None

if os.path.exists(PROPOSED_DIR) and len(os.listdir(PROPOSED_DIR)) > 0:
    print('Loading proposed scenario...')
    discovered_proposed = discover_files(PROPOSED_DIR)
    pairs_data_proposed = []

    for pair, files in discovered_proposed.items():
        xlsx = files.get('xlsx')
        csv = files.get('csv')
        if not xlsx:
            continue
        pairs_data_proposed.append(load_pair(
            name=pair,
            xlsx_path=xlsx,
            csv_path=csv,
        ))

    if pairs_data_proposed:
        proposed_result = run_simulation(
            pairs_data=pairs_data_proposed,
            initial_balance=INITIAL_BALANCE,
            scale_exponent=SCALE_EXPONENT,
            min_scale=MIN_SCALE,
            max_scale=MAX_SCALE,
            margin_requirements=margin_requirements,
        )
        proposed_summary = apply_shared_dd(proposed_result['summary'], proposed_result['curve_rows'])
        proposed_df_curve = pd.DataFrame(proposed_result['curve_rows'])
        proposed_df_curve['time'] = pd.to_datetime(proposed_df_curve['time'], format='%Y.%m.%d %H:%M')
        print(f"Proposed loaded: {proposed_summary['total_deals']} deals\n")


def calc_monthly_growth(df_in):
    """Average month-over-month balance growth %"""
    if df_in is None or 'balance' not in df_in.columns or 'time' not in df_in.columns:
        return None
    df = df_in.copy()
    df['ym'] = df['time'].dt.to_period('M')
    monthly_bal = df.groupby('ym')['balance'].last()
    if len(monthly_bal) < 2:
        return None
    returns = monthly_bal.pct_change().dropna()
    return returns.mean() * 100 if len(returns) > 0 else None


def calc_min_margin(df_in):
    """Minimum MT4/MT5 margin level %: equity / used_margin * 100."""
    if df_in is None:
        return None

    if 'margin_level_percent' in df_in.columns:
        margin = pd.to_numeric(df_in['margin_level_percent'], errors='coerce')
        margin = margin.replace([np.inf, -np.inf], np.nan).dropna()
        return margin.min() if len(margin) > 0 else None

    if 'used_margin' in df_in.columns and 'equity' in df_in.columns:
        used = pd.to_numeric(df_in['used_margin'], errors='coerce')
        eq = pd.to_numeric(df_in['equity'], errors='coerce')
        margin = (eq / used.replace(0, np.nan) * 100).replace([np.inf, -np.inf], np.nan).dropna()
        return margin.min() if len(margin) > 0 else None

    return None


baseline_monthly = calc_monthly_growth(baseline_df_curve)
baseline_margin = calc_min_margin(baseline_df_curve)
proposed_monthly = calc_monthly_growth(proposed_df_curve)
proposed_margin = calc_min_margin(proposed_df_curve)

comp = {
    'Metric': [
        'Initial Balance', 'Final Balance', 'Total Return %', 'CAGR %',
        'Max Drawdown %', 'Avg Monthly Growth %', 'Min Margin Level %',
        'Total Deals', 'Period Start'
    ],
    'Baseline': [
        f"${baseline_summary['initial_balance']:,.0f}",
        f"${baseline_summary['final_balance']:,.0f}",
        f"{baseline_summary['total_return_percent']:.2f}%",
        f"{baseline_summary['cagr_percent']:.2f}%",
        f"{baseline_summary['max_drawdown_percent']:.2f}%",
        f"{baseline_monthly:.3f}%" if baseline_monthly is not None else 'N/A',
        f"{baseline_margin:.2f}%" if baseline_margin is not None else 'N/A',
        f"{baseline_summary['total_deals']}",
        baseline_summary['period_start'][:10],
    ]
}

if proposed_summary is not None:
    comp['Proposed'] = [
        f"${proposed_summary['initial_balance']:,.0f}",
        f"${proposed_summary['final_balance']:,.0f}",
        f"{proposed_summary['total_return_percent']:.2f}%",
        f"{proposed_summary['cagr_percent']:.2f}%",
        f"{proposed_summary['max_drawdown_percent']:.2f}%",
        f"{proposed_monthly:.3f}%" if proposed_monthly is not None else 'N/A',
        f"{proposed_margin:.2f}%" if proposed_margin is not None else 'N/A',
        f"{proposed_summary['total_deals']}",
        proposed_summary['period_start'][:10],
    ]
    comp['Difference'] = [
        '',
        f"{((proposed_summary['final_balance'] / baseline_summary['final_balance']) - 1) * 100:+.1f}%",
        f"{proposed_summary['total_return_percent'] - baseline_summary['total_return_percent']:+.2f}pp",
        f"{proposed_summary['cagr_percent'] - baseline_summary['cagr_percent']:+.2f}pp",
        f"{proposed_summary['max_drawdown_percent'] - baseline_summary['max_drawdown_percent']:+.2f}pp",
        f"{proposed_monthly - baseline_monthly:+.3f}pp" if (proposed_monthly is not None and baseline_monthly is not None) else 'N/A',
        f"{proposed_margin - baseline_margin:+.2f}pp" if (proposed_margin is not None and baseline_margin is not None) else 'N/A',
        '',
        '',
    ]

df_comp = pd.DataFrame(comp)
display(df_comp.style.hide(axis='index').set_caption('Strategy Comparison: Baseline vs Proposed'))

In [ ]:
# Quick verification: actual unclamped drawdown and minimum equity
baseline_dd = baseline_summary['max_drawdown_percent']
proposed_dd = proposed_summary['max_drawdown_percent'] if proposed_summary is not None else None

print(f"Baseline Max Drawdown (unclamped): {baseline_dd:.4f}%")
if proposed_dd is not None:
    print(f"Proposed Max Drawdown (unclamped): {proposed_dd:.4f}%")

baseline_raw_equity = pd.DataFrame(baseline_result['curve_rows'])['equity']
proposed_raw_equity = pd.DataFrame(proposed_result['curve_rows'])['equity'] if proposed_result is not None else None

print(f"\nBaseline minimum equity: ${baseline_raw_equity.min():,.2f}")
if proposed_raw_equity is not None:
    print(f"Proposed minimum equity: ${proposed_raw_equity.min():,.2f}")